# v1 — TF-IDF + Logistic Regression 초기 판정 baseline

이 파일은 실험 기록을 해당 버전 시점까지 포함한 누적 Colab 노트북입니다.
실행 출력과 학습 가중치는 포함하지 않습니다. 위에서부터 순서대로 실행하세요.


In [ ]:
!git clone https://github.com/hover-nlp/hover.git
%cd hover
!bash download_data.sh

In [ ]:
!find data/hover -maxdepth 2 -type f

In [ ]:
# train 데이터
import json

path = "data/hover/hover_train_release_v1.1.json"

with open(path, encoding="utf-8") as f:
  data = json.load(f)

print("샘플 수:", len(data)) # 18171
print("첫 번째 샘플:", data[0])

In [ ]:
from collections import Counter

print("필드:", list(data[0].keys()))

print("\nHop 수별 샘플 수")
print(Counter(row["num_hops"] for row in data))

print("\n라벨별 샘플 수")
print(Counter(row["label"] for row in data))

two_hop = [row for row in data if row["num_hops"] == 2]
print("\n2-hop 샘플 수:", len(two_hop))

print("\n2-hop 라벨별 샘플 수") # 9052
print(Counter(row["label"] for row in two_hop))
# SUPPORTED: 6496(약 71%), NOT_SUPPORTED: 2556(28.2%)

현재 라벨이 균형적이지 않습니다. 따라서 baseline 평가에서는 Accuracy만 보지 말고 Macro-F1도 같이 보는게 좋습니다. 저분 SUPPORTED 라고 예측해도 Accuracy가 약 71.8% 나올 수 있으니 그보다 나은 모델인지 비교할 기준이 필요하다

In [ ]:
# Dev 데이터
import json

path = "data/hover/hover_dev_release_v1.1.json"

with open(path, encoding="utf-8") as f:
  data = json.load(f)

print("샘플 수:", len(data)) # 4000
print("첫 번째 샘플:", data[0])

In [ ]:
from collections import Counter

print("필드:", list(data[0].keys()))

print("\nHop 수별 샘플 수")
print(Counter(row["num_hops"] for row in data))

print("\n라벨별 샘플 수")
print(Counter(row["label"] for row in data))

two_hop = [row for row in data if row["num_hops"] == 2]
print("\n2-hop 샘플 수:", len(two_hop))

print("\n2-hop 라벨별 샘플 수") # 1126
print(Counter(row["label"] for row in two_hop))
# SUPPORTED: 605(약 53.7%), NOT_SUPPORTED: 521(46.3%)

baseline 평가 때는 Accuracy뿐 아니라 Macro-F1과 라벨별 점수도 함께 기록하면 한 라벨에 치우친 영향을 볼 수 있습니다.

In [ ]:
print("Train 2-hop 예시의 supporting_facts:")
print(two_hop[0]["supporting_facts"])

# 분석
# 해당 주장의 정답 근거가 두 Wikipedia 문서에 하나씩 연결돼 있다는 뜻입니다.
# ['Life Goes On (Fergie song)', 2] 해당 문서의 2번 문장
# ['M.I.L.F. $', 1] 해당 문서의 1번 문장

Macro-F1은 각 라벨의 F1 점수를 각각 계산한 뒤, 그 점수들을 똑같은 비중은 평균을 낸 값

지금 train 데이터는 SUPPORETED가 약 72%, NOT_SUPPORTED가 약 28%라서 라벨이 한 쪽으로 치우져 있습니다.
이럴 때 정확도만 보면 모델이 다수 라벨인 SUPPORETED를 잘 맞힌 것처럼 보일 수 있습니다.

supporting_facts에는 문서 제목과 문장 번호만 있으니, 그 번호에 해당하는 실제 근거 문장 텍스트가 들어있는 Wikipedia 코퍼스 파일을 확보

In [ ]:
!find data/hover -maxdepth 3 -type f

In [ ]:
import json

path = "data/hover/tfidf_retrieved/train_tfidf_doc_retrieval_results.json"

with open(path, encoding="utf-8") as f:
  retrieved = json.load(f)

print("자료형:", type(retrieved))
print("샘플 수:", len(retrieved) if isinstance(retrieved, list) else "목록 아님")

if isinstance(retrieved, list):
  print(json.dumps(retrieved[0], ensure_ascii=False, indent=2))[:3000]
elif isinstance(retrieved, dict):
  first_key = next(iter(retrieved))
  print("첫 번째 키:", first_key)
  print(json.dumps(retrieved[first_key], ensure_ascii=False, indent=2))[:3000]

In [ ]:
sample = next((item for item in retrieved if item is not None), None)

if sample is None:
    print("출력할 샘플이 없습니다.")
else:
    print(json.dumps(sample, ensure_ascii=False, indent=2)[:3000])

Hover 원본에서 2-hop train 9052개로 학습하고, 2-hop dev 1,126개로 평가하여, baseline 결과는 두 부분으로 나눠 기록

1. 무선 검색 성능: 검색 후보에 정답 근거 문서가 들어 있는지 확인
2. 라벨 분류 성능: claim을 보고 SUPPORTED / NOT_SUPPORTED를 예측

In [ ]:
!wget -O wiki_wo_links.db "https://nlp.cs.unc.edu/data/hover/wiki_wo_links.db"

In [ ]:
!ls -lh wiki_wo_links.db

테이블 확인

In [ ]:
import sqlite3

conn = sqlite3.connect("wiki_wo_links.db")
cursor = conn.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print(cursor.fetchall())

행/열 확인

In [ ]:
cursor.execute("PRAGMA table_info(documents);")
print("열 정보")
print(cursor.fetchall())

cursor.execute("SELECT * FROM documents LIMIT 1;")
print("\n샘플 데이터:")
print(cursor.fetchone())

id: Wikikpedia 문서 제목

text: 해당 문서의 본문 텍스트

HoVer 샘플의 supporting_facts에 적힌 문서 제목이 DB에서 찾아지는지 확인하기

In [ ]:
# 근거 문장 번호에 해당하는 문장 본문까지 확인하기. DB 본문을 단순히 문장으로 쪼개면 HoVer가 사용한
# 문장 번호와 어긋날 수 있으니, 우선 출력 결과를보고 본문이 어떤 형식으로 저장됐는지 확인
sample = two_hop[0]

print("Claim:", sample["claim"])
print("Label:", sample["label"])
print("Supporthin fatch:", sample["supporting_facts"])

for title, sentence_id in sample["supporting_facts"]:
  cursor.execute(
      "SELECT text FROM documents WHERE id = ?", (title,)
  )
  result = cursor.fetchone()

  print(f"\n문서: {title} / 근거 문장 번호: {sentence_id}")

  if result is None:
    print("DB에서 문서를 찾지 못함")
  else:
    print(result[0][:1000])

for title, sentence_id in sample["supporting_facts"]:
    cursor.execute(
        "SELECT text FROM documents WHERE id = ?",
        (title,)
    )
    text = cursor.fetchone()[0]

    print(f"\n문서: {title}")
    print(f"요청한 근거 문장 번호: {sentence_id}")
    print("줄바꿈 개수:", text.count("\n"))
    print("본문 앞부분:", repr(text[:1500]))

TF-IDF 결과 사용: 이미 계산된 후보 문서를 그대로 이용해 빠르게 시작할 수 있다.

TF-IDF 검색 구현: Wikipedia DB 전체를 검색 대상으로 삼아 문서를 찾는 단계부터 만들 수 있으나 구현과 계산이 더 필요함

BM25: 단어 겹침 기반의 또 다른 대표적인 검색 방법. 비교적 간단하고 검색에 자주 쓰임

Dense: 문장의 의미가 비슷한지를 임베딩으로 찾아. 표현이 달라도 관련 내용을 찾을 수 있지만, 임베딩 모델과 색인을 준비해야한다.

목표: HoVer의 2-hop claim을 입력받아 WikiPedia에서 근거 문장을 찾고, 그 문장을
바탕으로 SUPPORTED or NOT_SUPPORTED를 판정해 근거와 함께 보여준다.

claim: 시스템이 받아서 검증할 문장

label: 학습할 때 정답으로 알려주는 판정

supporting_facts: 정답 근거가 있는 문서 제목과 문장 번호. 실제 사용 시에는 주어지지 않고 검색 결과를 확인하는 답안표

Wikipedia DB의 id, text: 검색한 문서의 본문을 가져오는 자료

num_hops = 2: 이번 프로젝트에서 다룰 claim을 고르는 기준. 근거 문장이 반드시 두 개라는 뜻은 아님

In [ ]:
import json

TRAIN_PATH = "data/hover/hover_train_release_v1.1.json"

with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train = json.load(f)

print("원본 train 샘플 수:", len(train))
print("첫 번째 샘플 필드:", list(train[0].keys()))

In [ ]:
from sklearn.model_selection import train_test_split
from collections import Counter

# 2-hop 샘플만 선택
two_hop_data = [row for row in train if row["num_hops"] == 2]

# 학습용 80%, 검증용 20%로 분리
train_data, val_data = train_test_split(
    two_hop_data,
    test_size=0.2,
    random_state=42,
    stratify=[row["label"] for row in two_hop_data]
)

print("2-hop 전체:", len(two_hop_data))
print("학습용:", len(train_data))
print("검증용:", len(val_data))
print("학습 라벨 분포:", Counter(row["label"] for row in train_data))
print("검증 라벨 분포:", Counter(row["label"] for row in val_data))

In [ ]:
sample = train_data[0]

print("Claim:", sample["claim"])
print("Label:", sample["label"])
print("정답 근거 문서·문장:", sample["supporting_facts"])

In [ ]:
import sqlite3

DB_PATH = "/content/wiki_wo_links.db"  # 실제 파일 경로로 수정
conn = sqlite3.connect(DB_PATH)

tables = conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table'"
).fetchall()

print("연결한 DB:", DB_PATH)
print("테이블:", tables)

In [ ]:
!pip -q install blingfire

In [ ]:
from blingfire import text_to_sentences

for title, sentence_id in sample["supporting_facts"]:
    cursor.execute(
        "SELECT text FROM documents WHERE id = ?",
        (title,)
    )
    text = cursor.fetchone()[0]

    sentences = text_to_sentences(text).splitlines()

    print(f"\n문서: {title}")
    print(f"문장 수: {len(sentences)}")
    print(f"근거 문장 번호: {sentence_id}")

    if sentence_id < len(sentences):
        print("근거 문장:", sentences[sentence_id])
    else:
        print("이 번호에 해당하는 문장이 없음")

In [ ]:
cursor.execute("SELECT COUNT(*) FROM documents")
print("DB 문서 수:", cursor.fetchone()[0])

cursor.execute("SELECT id FROM documents LIMIT 5")
print("문서 제목 예시:")
for row in cursor.fetchall():
    print("-", row[0])

In [ ]:
# DB에 문서 제목만 읽어 들여서 claim의 단어와 겹치는 제목이 있는지 간단히 확인 1-hop
cursor.execute("SELECT id FROM documents LIMIT 10")
print([row[0] for row in cursor.fetchall()])

DB에서 문서 제목을 가져올 수 있음을 확인. 하지만 전체 523만 개 문서를 Python 메모리에 올리면 부담이 커질 수 있으니, 우선 DB에서 제목을 직접 검색하는 방식 수용

다만 TF-IDF 검색을 하려면 제목뿐 아니라 문서 본문도 비교 대상이어야 하기에 전체 문서를 바로 메모리에 올리지 말고 우선 샘플 claim의 핵심 단어가 들어간 문서 제목이 DB에 있는지 확인해보자

In [ ]:
keywords = ["Fergie", "Life Goes On", "M.I.L.F."]

for keyword in keywords:
    cursor.execute(
        "SELECT id FROM documents WHERE id LIKE ? LIMIT 20",
        (f"%{keyword}%",)
    )
    titles = [row[0] for row in cursor.fetchall()]
    print(f"\n[{keyword}] 제목 검색 결과 ({len(titles)}개)")
    for title in titles:
        print("-", title)

샘플의 정답 문서인 Life Goes On (Fergie song)과 M.I.L.F. $도 제목 검색 결과에 존재

다만 이건 claim 전체를 검색한 게 아니라, 내가 지정한 키워드가 문서 제목에 들어 있는지 확인한 것 그래서 이 결과만으로 TF-IDF 검색이나 Hop 1 검색이 된 건 아님

In [ ]:
# 문서의 문장들을 읽어보면서 claim의 어떤 부분을 뒷받침하는지 살펴 보는 것
title = "Life Goes On (Fergie song)"

cursor.execute(
    "SELECT text FROM documents WHERE id = ?",
    (title,)
)
text = cursor.fetchone()[0]

sentences = text_to_sentences(text).splitlines()

print("문서 제목:", title)
for i, sentence in enumerate(sentences):
    print(f"{i}: {sentence}")

caim과 관련된 문장을 확인

문장 2는 Life Goes On이 M.I.L.F. $를 뒤따라 발매된 싱글이라는 내용을 담고 있어.
하지만 claim에는 M.I.L.F. $가 Polow da Don에 의해 제작됐다는 내용도 있어. 이 문서는 그 부분을 말해주지 않으니, 다른 문서도 확인해야 해.

In [ ]:
title = "M.I.L.F. $"

cursor.execute(
    "SELECT text FROM documents WHERE id = ?",
    (title,)
)
text = cursor.fetchone()[0]

sentences = text_to_sentences(text).splitlines()

print("문서 제목:", title)
for i, sentence in enumerate(sentences):
    print(f"{i}: {sentence}")

이 예시에서 확인한 흐름을 일반화해 baseline의 Hop 1 검색 규칙을 정하는 것이야. 매 claim마다 사람이 제목을 넣는 대신, claim만 주었을 때 DB에서 후보 문서 제목을 자동으로 찾아야 해. 그때 쓸 검색 방식으로는 우선 제목을 대상으로 한 TF‑IDF 검색을 정하고, 몇 개의 후보 문서를 가져올지 결정하면 돼.

In [ ]:
import psutil

memory = psutil.virtual_memory()
print(f"사용 가능한 메모리: {memory.available / (1024**3):.1f} GB")

49 GB면 제목 검색 실험을 진행할 메모리는 충분해 보여. 다만 제목 523만 개로 TF‑IDF 색인을 만드는 작업은 시간이 걸릴 수 있으니, 먼저 전체 제목을 메모리에 읽어오는 단계만 하자.

In [ ]:
import json

RETRIEVAL_PATH = (
    "data/hover/tfidf_retrieved/"
    "train_tfidf_doc_retrieval_results.json"
)

with open(RETRIEVAL_PATH, "r", encoding="utf-8") as f:
    retrieval_data = json.load(f)

print("검색 결과 행 수:", len(retrieval_data))

example = retrieval_data[0]
print("claim:", example["claim"])
print("상위 문서 후보:", example["doc_retrieval_results"][0][0][:3])

KeyboardInterrupt는 오류라기보다, 523만 개 제목을 한 번에 가져오는 작업이 오래 걸려 중단된 거야. 메모리가 49GB여도 DB에서 대량으로 읽는 데 시간이 걸릴 수 있어. 방금 실행이 멈췄다면 titles 목록은 완성되지 않았을 테니, 전체 제목을 한 번에 가져오는 방법은 쓰지 말자.
그리고 앞서 내가 제목 전체를 먼저 메모리에 올리자고 한 건 좋은 안내가 아니었어. HoVer에서 제공하는 사전 계산 TF‑IDF 결과 파일에 claim별 상위 100개 후보가 있으니, 그 파일을 Hop 1 후보로 재사용하면 돼. 여기서 상위 3개만 선택하면 네가 정한 Hop 1 설계와 맞아.
단, 이 파일은 claim으로 한 번 검색한 결과라서 Hop 2 검색 결과까지 제공하는 건 아니야. Hop 2는 나중에 첫 단계에서 찾은 정보를 이용해 별도로 구현해야 해.
다음 단계에서는 Colab에서 기존 TF‑IDF 결과 파일의 경로와 샘플 한 건의 후보 문서 제목을 확인하자. 이미 파일을 읽어 retrieved 같은 변수에 담아둔 적이 있으니, 런타임 변수명을 추측하지 않고 파일에서 직접 확인하는 코드부터 안내할게.

파일을 읽었고, 첫 번째 claim에 대해 TF‑IDF가 뽑은 문서 후보 3개도 확인했어.
다음에는 이 후보 목록이 방금 확인한 train_data와 같은 claim에 연결되는지 확인하자. 파일의 첫 행은 데이터셋 순서상 같은 샘플이라고 단정하면 안 되니까, claim 문장으로 검색해 연결하는 게 안전해.

In [ ]:
target_claim = train_data[0]["claim"]

retrieved_example = next(
    row for row in retrieval_data
    if row["claim"] == target_claim
)

print("claim 일치:", retrieved_example["claim"] == target_claim)
print("상위 문서 후보:")
print(retrieved_example["doc_retrieval_results"][0][0][:3])

claim 일치: True라서, 이 샘플에 맞는 검색 결과를 찾았어. 후보 3개에 정답 문서 두 개도 들어 있어.
- Fas'h mosque
- Al-Masjid an-Nabawi
다만 이 후보 목록은 주장을 한 번 검색해 얻은 문서 순위야. 지금 한 일은 그중 상위 3개를 확인한 것이고, 시스템이 Hop 1에서 첫 근거를 찾거나 Hop 2에서 새로 검색한 건 아직 아니야.
다음에는 HoVer가 제공한 검색 결과를 우리 80:20 분할에 맞춰 사용할 수 있는지 확인하자. 같은 방식으로 train_data에서 claim 하나를 골라, 그 claim의 검색 결과 상위 3개와 정답 근거 문서 제목을 나란히 출력해 볼 거야. 이건 검색 후보 확인이며 아직 최종 성능 평가가 아니야.

In [ ]:
sample = train_data[0]

retrieved_example = next(
    row for row in retrieval_data
    if row["claim"] == sample["claim"]
)

top3_titles = retrieved_example["doc_retrieval_results"][0][0][:3]
gold_titles = [title for title, sentence_id in sample["supporting_facts"]]

print("Claim:", sample["claim"])
print("\n검색 상위 3개:")
for title in top3_titles:
    print("-", title)

print("\n정답 근거 문서:")
for title in gold_titles:
    print("-", title)

print("\n상위 3개에 정답 문서가 모두 포함됐나:",
      set(gold_titles).issubset(set(top3_titles)))

이 샘플에서는 검색 상위 3개 안에 정답 근거 문서 두 개가 모두 들어 있어. 좋은 예시지만, 샘플 하나만으로 전체 검색 성능을 판단할 수는 없어.
다음 단계에서는 이 확인을 검증용 1,811개 전체에 반복해 상위 3개 문서가 정답 근거를 얼마나 자주 포함하는지 측정하자. 이 수치는 검색 단계 평가야. 아직 문장 검색이나 SUPPORTED/NOT_SUPPORTED 판정 성능은 아니야.

그럼 검증용 1,811개에서 검색 상위 3개가 정답 근거 문서를 얼마나 포함하는지 계산하자. 아래 코드를 실행해 줘. 이 코드는 정답 근거를 검색 입력으로 쓰지 않고, 검색 결과와 나중에 비교해서 점수만 계산해.

In [ ]:
# claim을 기준으로 사전 검색 결과를 찾기 위한 색인
retrieval_by_claim = {
    row["claim"]: row
    for row in retrieval_data
}

total = 0
any_gold_found = 0
all_gold_found = 0
missing_claims = 0

for sample in val_data:
    retrieved = retrieval_by_claim.get(sample["claim"])

    if retrieved is None:
        missing_claims += 1
        continue

    top3_titles = set(retrieved["doc_retrieval_results"][0][0][:3])
    gold_titles = {title for title, sentence_id in sample["supporting_facts"]}

    total += 1
    if top3_titles & gold_titles:
        any_gold_found += 1
    if gold_titles.issubset(top3_titles):
        all_gold_found += 1

print("검증 claim 수:", len(val_data))
print("검색 결과와 연결된 claim 수:", total)
print("연결되지 않은 claim 수:", missing_claims)
print(f"정답 문서를 하나 이상 찾음: {any_gold_found}/{total} ({any_gold_found/total:.1%})")
print(f"정답 문서를 전부 찾음: {all_gold_found}/{total} ({all_gold_found/total:.1%})")

결과가 나왔어. 검증용 claim 1,811개 모두 검색 결과와 연결됐어.
- **82.3%**에서는 정답 근거 문서 중 하나 이상이 상위 3개에 들어 있었어.
- **32.5%**에서는 정답 근거 문서가 모두 상위 3개에 들어 있었어.
이건 문서 검색만 평가한 결과야. 아직 정답 문장을 골랐거나 SUPPORTED/NOT_SUPPORTED를 예측한 성능은 아니야. 특히 제공된 TF‑IDF 목록은 claim으로 한 번 검색한 결과라서, 상위 3개에서 근거 문서를 놓친 경우 Hop 2 검색으로 보완할 여지가 있어.
이제 다음 단계로 넘어가자. Hop 1 후보 문서의 문장들을 읽고, claim을 확인할 근거가 있는지 살펴보는 것이야. 아래 코드는 검증 샘플 하나에 대해 상위 3개 문서의 문장들을 출력해. supporting_facts는 출력에 쓰지 않고, 검색 후보 문서의 본문만 확인해.

In [ ]:
sample = val_data[0]

retrieved_example = retrieval_by_claim[sample["claim"]]
top3_titles = retrieved_example["doc_retrieval_results"][0][0][:3]

print("Claim:", sample["claim"])

for title in top3_titles:
    cursor.execute(
        "SELECT text FROM documents WHERE id = ?",
        (title,)
    )
    result = cursor.fetchone()

    print(f"\n문서: {title}")

    if result is None:
        print("DB에서 문서를 찾지 못함")
        continue

    sentences = text_to_sentences(result[0]).splitlines()
    for i, sentence in enumerate(sentences):
        print(f"{i}: {sentence}")

이 예시는 Hop 2가 왜 필요한지 보여줘.
Hop 1의 Fanny Talbot 문서에서 Fanny Talbot이 미술 평론가 John Ruskin과 친구이자 서신을 주고받았다는 연결 정보를 찾았어. 그런데 상위 3개 문서에는 John Ruskin 문서가 없고, claim의 나머지 내용과 직접 관련이 적어 보이는 Brian Sewell 문서가 들어 있어. 그러니 다음 단계에서는 Hop 1에서 얻은 이름 John Ruskin을 이용해 관련 문서를 찾아야 해.
우선 DB에 John Ruskin 문서가 있는지 확인하고, 본문을 읽어보자. 아래 코드를 실행해 줘.

In [ ]:
title = "John Ruskin"

cursor.execute(
    "SELECT text FROM documents WHERE id = ?",
    (title,)
)
result = cursor.fetchone()

if result is None:
    print("문서를 찾지 못함:", title)
else:
    sentences = text_to_sentences(result[0]).splitlines()
    print("문서 제목:", title)
    for i, sentence in enumerate(sentences):
        print(f"{i}: {sentence}")

이건 Hop 2의 검색 대상 문서가 DB에 있고 claim의 나머지 내용을 설명하는지 살펴보는 예시야. 아직 이름을 자동으로 추출하거나 Hop 2 검색을 자동화한 건 아니야.

이 문장으로 claim의 연결이 확인돼. Fanny Talbot 문서에서 찾은 이름인 John Ruskin의 문서가 “Victorian era의 leading English art critic”이라고 설명하고 있어. 이 예시는 Hop 1에서 연결 인물을 찾고, Hop 2에서 그 인물의 문서를 찾아 claim의 나머지 부분을 확인하는 흐름에 맞아.
이제 다음 단계는 사람이 직접 입력한 "John Ruskin"을 시스템이 자동으로 찾아내도록 만드는 거야. 간단한 baseline 규칙은 다음과 같아.
1. Hop 1 문서에서 claim과 관련 있는 문장을 고른다.
2. 그 문장에서 인물·장소·기관 이름을 추출한다.
3. claim에 이미 나온 이름은 제외하고, 새 이름을 Hop 2 검색어로 쓴다.
4. 그 이름과 일치하는 Wikipedia 문서를 찾아 읽는다.
먼저 이 예시에서 2번을 시험해 볼 수 있어. 문장 하나에서 인물 이름을 뽑는 도구로 spaCy를 사용하자. Colab 셀에서 아래를 실행해 줘.

In [ ]:
!pip -q install spacy
!python -m spacy download en_core_web_sm

좋아. 먼저 Fanny Talbot 문장에서 spaCy가 어떤 이름을 인식하는지 확인하자. 새 셀에서 실행해 줘.

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")

claim = "The man associated with Fanny Talbot was the leading art critic of the Victorian era in Great Britain."
sentence = (
    "Fanny Talbot (née) Browne (1824–1917) was a landowner and philanthropist, "
    "and a friend and correspondent of the influential art critic John Ruskin."
)

doc = nlp(sentence)

print("인식한 이름:")
for entity in doc.ents:
    print(entity.text, "→", entity.label_)

spaCy가 인물 이름 두 개를 찾았어.
- Fanny Talbot — claim에 이미 있는 이름
- John Ruskin — claim에 나오지 않은 새 인물 이름
따라서 이 예시에서는 claim에 이미 나온 이름을 제외하면 **Hop 2 검색어 후보는 John Ruskin**이야. 1824–1917은 숫자 정보라 검색어 후보에서 제외하면 돼.
이제 이 규칙을 코드로 적용해 보자. 실행하면 추출된 인물 중 claim에 없는 이름만 출력돼.

In [ ]:
claim_lower = claim.lower()

hop2_candidates = [
    entity.text
    for entity in doc.ents
    if entity.label_ == "PERSON"
    and entity.text.lower() not in claim_lower
]

print("Hop 2 검색어 후보:", hop2_candidates)

In [ ]:
for query in hop2_candidates:
    cursor.execute(
        "SELECT id, text FROM documents WHERE lower(id) = lower(?)",
        (query,)
    )
    result = cursor.fetchone()

    print(f"\n검색어: {query}")

    if result is None:
        print("일치하는 문서 제목을 찾지 못함")
        continue

    title, text = result
    print("찾은 문서:", title)

    sentences = text_to_sentences(text).splitlines()
    for i, sentence in enumerate(sentences):
        print(f"{i}: {sentence}")

좋아. 이제 이 예시에서는 Hop 1 문장에서 John Ruskin을 뽑고, 그 이름으로 Hop 2 문서를 찾는 흐름까지 연결됐어.
다만 지금까지는 Fanny Talbot 문장을 사람이 직접 골라 spaCy에 넣었어. 실제 baseline은 claim과 상위 3개 문서만 받고, Hop 1에서 어떤 문장을 근거 후보로 고를지도 자동으로 해야 해. 다음은 TF‑IDF로 claim과 후보 문장들의 단어 유사도를 비교해 가장 관련 높은 문장을 찾아보는 단계야.
아래 코드를 실행해 줘. supporting_facts는 사용하지 않고, 검색 상위 3개 문서의 본문만 대상으로 점수를 매겨.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

sample = val_data[0]
retrieved_example = retrieval_by_claim[sample["claim"]]
top3_titles = retrieved_example["doc_retrieval_results"][0][0][:3]

candidates = []

for title in top3_titles:
    cursor.execute(
        "SELECT text FROM documents WHERE id = ?",
        (title,)
    )
    result = cursor.fetchone()

    if result is None:
        continue

    sentences = text_to_sentences(result[0]).splitlines()
    for sentence_id, sentence in enumerate(sentences):
        candidates.append((title, sentence_id, sentence))

# claim과 후보 문장들을 같은 TF-IDF 공간에 놓고 유사도 계산
texts = [sample["claim"]] + [item[2] for item in candidates]
matrix = TfidfVectorizer(stop_words="english").fit_transform(texts)
scores = cosine_similarity(matrix[0:1], matrix[1:]).ravel()

ranked = sorted(
    zip(scores, candidates),
    key=lambda item: item[0],
    reverse=True
)

print("Claim:", sample["claim"])
print("\n관련 문장 상위 5개:")
for score, (title, sentence_id, sentence) in ranked[:5]:
    print(f"\n점수: {score:.3f} | 문서: {title} | 문장: {sentence_id}")
    print(sentence)

상위 문장이 Fanny Talbot과 John Ruskin의 관계를 설명하는 문장인지 확인해 보자. 이 점수는 문장 후보를 순위 매기는 기준이지, 그 문장이 사실을 입증한다고 판정하는 점수는 아니야.

결과를 보면 중요한 한계가 드러났어. TF‑IDF 점수가 가장 높은 문장이 정답 근거 문장인 건 아니야.
- 1위 Brian Sewell 문장은 “art critic”이라는 표현이 claim과 겹쳐 점수가 높았지만, Fanny Talbot과의 관계는 설명하지 않아.
- 2위 Fanny Talbot 문장은 점수는 조금 낮아도 John Ruskin과의 관계를 설명해.
그래서 baseline이 1위 문장만 사용하면 연결 정보를 놓칠 수 있어. 다음 단계에서는 상위 3개 문장에서 인물 이름을 추출해 Hop 2 검색 후보를 만들자. 이렇게 하면 Brian Sewell과 John Ruskin이 모두 후보에 포함될 수 있어. 후보가 맞는지는 Hop 2 문서와 claim을 함께 확인해야 해.
아래 코드를 실행해 줘.

In [ ]:
claim_lower = sample["claim"].lower()
hop2_candidates = []

print("상위 3개 문장에서 추출한 인물 후보:")

for score, (title, sentence_id, sentence) in ranked[:3]:
    doc = nlp(sentence)

    people = [
        ent.text
        for ent in doc.ents
        if ent.label_ == "PERSON"
        and ent.text.lower() not in claim_lower
    ]

    print(f"\n문서: {title} | 문장: {sentence_id} | 점수: {score:.3f}")
    print(sentence)
    print("새 인물 이름:", people)

    hop2_candidates.extend(people)

# 중복 제거
hop2_candidates = list(dict.fromkeys(hop2_candidates))
print("\nHop 2 검색 후보 전체:", hop2_candidates)

이 결과로 문장 상위 3개에서 이름 후보가 만들어지는지 확인하자. 이름이 후보로 나왔다는 것만으로 그 인물이 claim의 정답이라는 뜻은 아니야.

좋아. 상위 3개 문장에서 John Ruskin과 Earl Talbot 두 이름이 후보로 나왔어. 여기서 Earl Talbot은 문맥상 잡음일 수 있어. 이름을 추출하는 것만으로는 Hop 2 정답을 고를 수 없으니, 각 후보 문서를 조회한 뒤 claim과 관련된 문장을 다시 순위 매기자.
아래 코드는 Hop 2 후보 문서를 DB에서 찾고, 각 문장을 원래 claim과 TF‑IDF로 비교해 상위 5개를 보여줘.

In [ ]:
hop2_sentences = []

for title in hop2_candidates:
    cursor.execute(
        "SELECT id, text FROM documents WHERE lower(id) = lower(?)",
        (title,)
    )
    result = cursor.fetchone()

    if result is None:
        print(f"문서를 찾지 못함: {title}")
        continue

    doc_title, text = result
    sentences = text_to_sentences(text).splitlines()

    for sentence_id, sentence in enumerate(sentences):
        hop2_sentences.append((doc_title, sentence_id, sentence))

texts = [sample["claim"]] + [item[2] for item in hop2_sentences]
matrix = TfidfVectorizer(stop_words="english").fit_transform(texts)
scores = cosine_similarity(matrix[0:1], matrix[1:]).ravel()

ranked_hop2 = sorted(
    zip(scores, hop2_sentences),
    key=lambda item: item[0],
    reverse=True
)

print("Hop 2 문장 후보 상위 5개:")
for score, (title, sentence_id, sentence) in ranked_hop2[:5]:
    print(f"\n점수: {score:.3f} | 문서: {title} | 문장: {sentence_id}")
    print(sentence)

John Ruskin 문서의 첫 문장이 높은 순위로 나오면, 이 샘플에서는 두 번째 근거를 찾는 흐름이 동작한 거야. 점수 순위는 후보를 정렬할 뿐, 그 자체로 사실 여부를 판정하진 않아.

좋아. Hop 2에서 claim의 나머지 부분을 직접 설명하는 문장을 찾았어. 이 샘플의 검색 경로는 이제 이렇게 연결돼.
1. Hop 1: Fanny Talbot 문장에서 John Ruskin과의 관계를 찾음
2. Hop 2: John Ruskin 문서에서 Victorian era의 leading art critic이었다는 문장을 찾음
Earl Talbot도 후보로 나왔지만, claim을 설명하는 근거로는 적절하지 않아 보여. 그래서 다음은 두 홉에서 찾은 근거 문장과 데이터의 정답 라벨을 함께 출력해 결과 형식을 확인하자.

In [ ]:
# Hop 1 후보 중 Hop 2 인물 이름이 실제로 언급된 문장을 고름
hop1_evidence = None

for score, (title, sentence_id, sentence) in ranked:
    if any(name.lower() in sentence.lower() for name in hop2_candidates):
        hop1_evidence = (title, sentence_id, sentence)
        break

print("Claim:", sample["claim"])
print("데이터 정답 라벨:", sample["label"])

if hop1_evidence:
    title, sentence_id, sentence = hop1_evidence
    print(f"\nHop 1 근거 후보 | {title} | 문장 {sentence_id}")
    print(sentence)

if ranked_hop2:
    score, (title, sentence_id, sentence) = ranked_hop2[0]
    print(f"\nHop 2 근거 후보 | {title} | 문장 {sentence_id}")
    print(sentence)

이 출력에서 두 근거가 claim의 서로 다른 부분을 설명하는지 확인할 거야. 여기까지는 근거를 찾는 과정이고, SUPPORTED/NOT_SUPPORTED를 자동으로 판정하는 모델은 아직 만들지 않았어.

여기서 중요한 불일치가 보여. 우리가 찾은 두 문장은 claim을 뒷받침하는 것처럼 읽히는데, 데이터 라벨은 NOT_SUPPORTED야. 아직 라벨 오류라고 단정하지 말고, HoVer가 이 샘플에 지정한 정답 근거를 확인하자. 검색기가 다른 인물을 연결했거나, claim의 일부를 놓쳤는지 비교해야 해.
아래 코드는 supporting_facts에 적힌 문서와 문장 번호를 출력해. 그 정답 근거를 방금 찾은 Hop 1·Hop 2 문장과 비교해 보자.

In [ ]:
print("라벨:", sample["label"])
print("HoVer 정답 근거:", sample["supporting_facts"])

for title, sentence_id in sample["supporting_facts"]:
    cursor.execute(
        "SELECT text FROM documents WHERE id = ?",
        (title,)
    )
    result = cursor.fetchone()

    print(f"\n문서: {title} | 근거 문장 번호: {sentence_id}")

    if result is None:
        print("DB에서 문서를 찾지 못함")
        continue

    sentences = text_to_sentences(result[0]).splitlines()

    if sentence_id < len(sentences):
        print(sentences[sentence_id])
    else:
        print(f"문장 번호를 찾지 못함 (문서 문장 수: {len(sentences)})")

확인했어. 검색 실패는 아니야. 우리가 찾은 두 문장이 HoVer의 정답 근거와 정확히 같아.
그런데 이 근거들은 claim을 뒷받침하는 것처럼 읽히는데 라벨은 NOT_SUPPORTED야. 그래서 이 샘플은 라벨이나 claim 해석을 따로 검토해야 하는 사례로 표시해 두자. 한 샘플만 보고 데이터 라벨이 틀렸다고 단정하진 말자.
baseline을 만들 때는 두 가지를 구분해야 해.
- 학습·평가의 정답 라벨은 우선 데이터에 기록된 NOT_SUPPORTED를 그대로 사용해.
- 사람이 읽었을 때 근거와 라벨이 맞지 않아 보이는 사례는 별도로 기록해. 평가 결과를 해석할 때 이 점을 고려하면 돼.
그리고 supporting_facts는 해당 claim과 관련해 데이터셋이 지정한 근거 문장이라는 뜻이야. 그 근거가 항상 라벨을 직관적으로 설명해 주는 건 아닐 수 있어.
현재까지 확인한 것: Hop 1과 Hop 2에서 정답 근거 문장들을 찾았지만, 이 샘플의 데이터 라벨과 근거 해석 사이에는 의문이 남아 있어. 다음 단계는 이 근거 문장들을 claim과 함께 판정 모델에 입력하는 형태로 정리하는 거야.

좋아. 다음 단계는 판정 모델이 받을 입력 형식을 정하는 거야. 모델에는 claim과 두 홉에서 찾은 문장을 넣고, 학습 목표로 데이터의 라벨을 사용해.
먼저 지금 샘플 하나를 그 형식으로 만들어 출력해 보자. 이 코드는 모델을 학습시키지는 않아.

In [ ]:
hop1_title, hop1_sentence_id, hop1_sentence = hop1_evidence
hop2_score, (hop2_title, hop2_sentence_id, hop2_sentence) = ranked_hop2[0]

model_input = (
    f"Claim: {sample['claim']}\n"
    f"Evidence 1: {hop1_sentence}\n"
    f"Evidence 2: {hop2_sentence}"
)

target_label = sample["label"]

print("모델 입력 예시:")
print(model_input)
print("\n학습 정답 라벨:", target_label)

이 샘플로 모델 입력 형식은 확인했어. 입력은 Claim + Evidence 1 + Evidence 2, 정답은 데이터 라벨인 NOT_SUPPORTED야. 다만 근거와 라벨이 맞지 않아 보이는 점은 검토 사례로 기록해 두고, 라벨은 임의로 바꾸지 말자.
아직 샘플 하나만 만들었으니 모델을 학습할 수는 없어. 다음은 방금 수동으로 했던 Hop 1 후보 문장 순위 매기기와 새 인물 이름 추출을 함수로 묶어, 같은 처리를 자동화할 준비를 하는 단계야. 우선 현재 샘플에 함수가 제대로 적용되는지 확인하자.

In [ ]:
def find_hop1_candidates(example, sentence_top_k=3):
    retrieved = retrieval_by_claim.get(example["claim"])
    if retrieved is None:
        return [], [], []

    doc_titles = retrieved["doc_retrieval_results"][0][0][:3]
    sentence_candidates = []

    for title in doc_titles:
        cursor.execute(
            "SELECT text FROM documents WHERE id = ?",
            (title,)
        )
        result = cursor.fetchone()

        if result is None:
            continue

        sentences = text_to_sentences(result[0]).splitlines()
        for sentence_id, sentence in enumerate(sentences):
            sentence_candidates.append((title, sentence_id, sentence))

    if not sentence_candidates:
        return doc_titles, [], []

    texts = [example["claim"]] + [item[2] for item in sentence_candidates]
    matrix = TfidfVectorizer(stop_words="english").fit_transform(texts)
    scores = cosine_similarity(matrix[0:1], matrix[1:]).ravel()

    ranked_sentences = sorted(
        zip(scores, sentence_candidates),
        key=lambda item: item[0],
        reverse=True
    )

    claim_lower = example["claim"].lower()
    new_people = []

    for _, (_, _, sentence) in ranked_sentences[:sentence_top_k]:
        for entity in nlp(sentence).ents:
            if (
                entity.label_ == "PERSON"
                and entity.text.lower() not in claim_lower
                and entity.text not in new_people
            ):
                new_people.append(entity.text)

    return doc_titles, ranked_sentences[:sentence_top_k], new_people


doc_titles, hop1_top_sentences, hop2_candidates = find_hop1_candidates(sample)

print("Hop 1 문서:", doc_titles)
print("\nHop 1 문장 후보:")
for score, (title, sentence_id, sentence) in hop1_top_sentences:
    print(f"\n점수: {score:.3f} | {title} | 문장 {sentence_id}")
    print(sentence)

print("\nHop 2 인물 후보:", hop2_candidates)

좋아. 자동화한 Hop 1에서도 John Ruskin이 Hop 2 후보로 나왔어. 동시에 Earl Talbot도 후보에 포함됐으니, Hop 2에서도 문장 점수로 후보를 정렬해야 해. 이 점수는 관련 문장을 고르는 기준이고, 사실 여부 판정은 아니야.
다음은 Hop 2 문서 조회와 문장 순위를 함수로 묶고, 현재 샘플에서 가장 높은 문장을 출력하는 단계야.

In [ ]:
def find_hop2_evidence(example, entity_candidates):
    sentence_candidates = []

    for entity in entity_candidates:
        cursor.execute(
            "SELECT id, text FROM documents WHERE lower(id) = lower(?)",
            (entity,)
        )
        result = cursor.fetchone()

        if result is None:
            continue

        title, text = result
        sentences = text_to_sentences(text).splitlines()

        for sentence_id, sentence in enumerate(sentences):
            sentence_candidates.append((title, sentence_id, sentence))

    if not sentence_candidates:
        return []

    texts = [example["claim"]] + [item[2] for item in sentence_candidates]
    matrix = TfidfVectorizer(stop_words="english").fit_transform(texts)
    scores = cosine_similarity(matrix[0:1], matrix[1:]).ravel()

    return sorted(
        zip(scores, sentence_candidates),
        key=lambda item: item[0],
        reverse=True
    )


hop2_ranked = find_hop2_evidence(sample, hop2_candidates)

print("Hop 2 문장 후보 상위 3개:")
for score, (title, sentence_id, sentence) in hop2_ranked[:3]:
    print(f"\n점수: {score:.3f} | {title} | 문장 {sentence_id}")
    print(sentence)

John Ruskin 문서의 첫 문장이 가장 위에 나오면, Hop 1에서 찾은 인물 후보를 Hop 2 문서 검색으로 연결하는 과정이 함수로 정리된 거야.

좋아. Hop 2에서도 John Ruskin의 문장 0이 가장 높은 점수를 받았어. 이제 함수 두 개의 결과를 합쳐 claim, Hop 1 근거, Hop 2 근거, 라벨을 한 샘플로 만드는 단계야.

In [ ]:
def build_model_example(example):
    doc_titles, hop1_top, entity_candidates = find_hop1_candidates(example)

    if not hop1_top:
        return None

    # Hop 2 검색어가 실제 언급된 Hop 1 문장을 연결 근거로 선택
    hop1_match = next(
        (
            item for item in hop1_top
            if any(name.lower() in item[1][2].lower()
                   for name in entity_candidates)
        ),
        hop1_top[0]
    )
    _, (hop1_title, hop1_sentence_id, hop1_sentence) = hop1_match

    hop2_ranked = find_hop2_evidence(example, entity_candidates)

    if hop2_ranked:
        _, (hop2_title, hop2_sentence_id, hop2_sentence) = hop2_ranked[0]
    else:
        hop2_title, hop2_sentence_id, hop2_sentence = "", -1, ""

    model_text = (
        f"Claim: {example['claim']}\n"
        f"Evidence 1: {hop1_sentence}\n"
        f"Evidence 2: {hop2_sentence}"
    )

    return {
        "text": model_text,
        "label": example["label"],
        "hop1_title": hop1_title,
        "hop1_sentence_id": hop1_sentence_id,
        "hop2_title": hop2_title,
        "hop2_sentence_id": hop2_sentence_id,
    }


model_example = build_model_example(sample)

print("라벨:", model_example["label"])
print("Hop 1 문서:", model_example["hop1_title"])
print("Hop 2 문서:", model_example["hop2_title"])
print("\n모델 입력:")
print(model_example["text"])

이 결과가 앞에서 수동으로 만든 입력과 비슷하면, 한 샘플의 검색 결과를 모델 입력 형식으로 자동 조립하는 것까지 된 거야. 아직 학습은 하지 않았어. 출력 결과를 확인한 뒤 이 함수를 데이터 전체에 적용할 거야.

한 샘플에 대해 검색 결과를 모델 입력 형식으로 자동 조립하는 것까지 확인했어. 이제 전체 7,241개를 바로 처리하기 전에, 학습 데이터 앞의 100개로 함수가 반복 실행되는지 시험하자. 이건 속도와 검색 누락을 확인하는 점검이라 아직 최종 성능 결과는 아니야.

In [ ]:
from tqdm.auto import tqdm
from collections import Counter

train_preview = []
errors = []

for row in tqdm(train_data[:100]):
    try:
        result = build_model_example(row)
        if result is not None:
            train_preview.append(result)
    except Exception as e:
        errors.append((row["uid"], str(e)))

print("처리한 샘플:", 100)
print("모델 입력 생성 성공:", len(train_preview))
print("오류:", len(errors))
print("Hop 2 문서까지 찾음:",
      sum(bool(row["hop2_title"]) for row in train_preview))
print("라벨 분포:",
      Counter(row["label"] for row in train_preview))

if errors:
    print("\n첫 오류:", errors[0])

100개 처리에 9분 7초가 걸렸어. 코드 오류는 없지만, 지금 속도대로 전체를 처리하면 학습 데이터만 약 9시간, 검증 데이터까지 약 11시간이 걸릴 수 있어. 전체 실행은 아직 하지 말자.
또 100개 중 Hop 2 문서까지 찾은 건 51개야. 나머지 49개는 코드 오류가 아니라 두 번째 문서를 못 찾은 경우야. 빈 Hop 2 근거를 그대로 두고 모델을 학습하면 결과가 왜곡될 수 있으니, 먼저 왜 문서를 못 찾았는지 확인하자.
아래 코드는 이미 처리한 100개 중 Hop 2 문서를 못 찾은 샘플 최대 5개만 살펴봐. 이 샘플들의 Hop 2 인물 후보가 DB 문서 제목과 일치하는지 확인하는 점검이야.

In [ ]:
missing_positions = [
    i for i, result in enumerate(train_preview)
    if not result["hop2_title"]
]

print("Hop 2 문서를 못 찾은 샘플:", len(missing_positions))

for i in missing_positions[:5]:
    row = train_data[i]
    _, _, people = find_hop1_candidates(row)

    print("\nClaim:", row["claim"])
    print("Hop 2 인물 후보:", people)

    for person in people:
        cursor.execute(
            "SELECT id FROM documents WHERE lower(id) = lower(?) LIMIT 1",
            (person,)
        )
        match = cursor.fetchone()
        print(f"  {person} →", match[0] if match else "일치하는 제목 없음")

출력 결과를 보면 Hop 2 누락이 이름 추출 문제인지, 문서 제목 일치 문제인지 구분할 수 있어. 이 원인을 확인한 뒤 처리 속도와 검색 누락을 함께 개선하자.

첫 번째 누락 샘플에서 원인이 보이기 시작했어. 현재 함수는 인물(PERSON)만 Hop 2 검색어로 인정해. 하지만 Fash Mosque claim은 Al-Masjid an-Nabawi처럼 장소나 종교 시설 이름을 연결 고리로 써야 할 수 있어. 인물만 찾도록 만든 규칙이 너무 좁았던 거야.
두 번째 샘플의 vlag van Noord-Holland는 후보 이름으로 나왔지만, 이 이름이 실제 claim의 연결 고리인지는 아직 확인되지 않았어. 이름 추출에는 이런 잡음 후보도 생길 수 있어.
우선 spaCy가 Fash 샘플의 연결 이름을 어떤 종류로 인식하는지 확인하자. 아래 셀을 실행해 줘.

In [ ]:
test_sentence = "It is 4.5 km from Al-Masjid an-Nabawi."

print("문장:", test_sentence)
print("인식한 항목:")
for entity in nlp(test_sentence).ents:
    print(entity.text, "→", entity.label_)

이 결과를 확인한 뒤 Hop 2 후보에 인물뿐 아니라 장소·기관도 포함하도록 규칙을 고치자. 지금의 51/100은 시험 실행에서 문서가 연결된 비율일 뿐, 최종 성능 수치는 아니야.

확인했어. spaCy가 Al-Masjid an-Nabawi 전체를 인식하지 못하고 Al-Masjid만 PERSON으로 잘못 분류했어. 이름 추출만으로 Hop 2 검색어를 만들면 이런 식으로 장소 이름을 놓치거나 잘못 처리할 수 있어.
이 경우는 추출된 이름을 그대로 검색하는 대신, 그 이름으로 시작하는 Wikipedia 문서 제목을 찾아보자. 아래 코드로 Al-Masjid로 시작하는 제목을 확인해 줘.

In [ ]:
prefix = "Al-Masjid"

cursor.execute(
    "SELECT id FROM documents WHERE id LIKE ? LIMIT 20",
    (prefix + "%",)
)

print("문서 제목 후보:")
for row in cursor.fetchall():
    print("-", row[0])

결과에 Al-Masjid an-Nabawi가 나오면, NER가 뽑은 짧은 이름을 DB 제목 후보로 확장하는 방법을 시험할 수 있어. 다만 다른 이름에도 항상 잘 맞는지는 별도로 확인해야 해.

좋아. Al-Masjid라는 짧은 이름으로도 DB에서 Al-Masjid an-Nabawi 제목을 찾을 수 있다는 걸 확인했어.
다만 아까 Fash 샘플의 Hop 2 후보가 비어 있었던 이유는 아직 확정되지 않았어. 연결 문장이 문장 순위 상위 3개에서 빠졌는지, 아니면 이름 추출 단계에서 빠졌는지 먼저 살펴보자. 해당 누락 샘플에서 상위 10개 문장과 spaCy가 인식한 이름을 출력해 줘.

In [ ]:
fash_row = train_data[missing_positions[0]]

_, top10_sentences, people = find_hop1_candidates(
    fash_row,
    sentence_top_k=10
)

print("Claim:", fash_row["claim"])

for score, (title, sentence_id, sentence) in top10_sentences:
    entities = [
        (entity.text, entity.label_)
        for entity in nlp(sentence).ents
    ]

    print(f"\n점수: {score:.3f} | {title} | 문장 {sentence_id}")
    print(sentence)
    print("인식한 이름:", entities)

print("\nHop 2 인물 후보:", people)

이제 Fash 샘플에서 Hop 2 후보가 비었던 원인을 찾았어. 두 가지가 겹쳤어.
- 함수가 claim에 이미 나온 이름을 후보에서 제외했어. 그런데 이 샘플에서는 Al-Masjid an-Nabawi가 claim에도 나오고, 다음 문서를 찾는 연결 이름이기도 해.
- spaCy는 그 이름을 Al-Masjid만 인식했어. 그래서 DB에서는 이름과 정확히 같은 제목이 아니라 그 이름으로 시작하는 제목도 찾아야 해.
먼저 find_hop1_candidates 함수에서 인물 후보를 모으는 부분을 찾아 아래처럼 바꿔 줘. claim에 이름이 이미 있는지 검사하는 조건을 없애는 게 핵심이야.

In [ ]:
def find_hop1_candidates(example, sentence_top_k=3):
    retrieved = retrieval_by_claim.get(example["claim"])
    if retrieved is None:
        return [], [], []

    doc_titles = retrieved["doc_retrieval_results"][0][0][:3]
    sentence_candidates = []

    for title in doc_titles:
        cursor.execute(
            "SELECT text FROM documents WHERE id = ?",
            (title,)
        )
        result = cursor.fetchone()

        if result is None:
            continue

        sentences = text_to_sentences(result[0]).splitlines()
        for sentence_id, sentence in enumerate(sentences):
            sentence_candidates.append((title, sentence_id, sentence))

    if not sentence_candidates:
        return doc_titles, [], []

    texts = [example["claim"]] + [item[2] for item in sentence_candidates]
    matrix = TfidfVectorizer(stop_words="english").fit_transform(texts)
    scores = cosine_similarity(matrix[0:1], matrix[1:]).ravel()

    ranked_sentences = sorted(
        zip(scores, sentence_candidates),
        key=lambda item: item[0],
        reverse=True
    )

    # claim에 이미 나온 이름도 연결 후보로 유지
    allowed_labels = {"PERSON", "ORG", "GPE", "LOC", "FAC"}
    entity_candidates = []

    for _, (_, _, sentence) in ranked_sentences[:sentence_top_k]:
        for entity in nlp(sentence).ents:
            if (
                entity.label_ in allowed_labels
                and entity.text not in entity_candidates
            ):
                entity_candidates.append(entity.text)

    return doc_titles, ranked_sentences[:sentence_top_k], entity_candidates

In [ ]:
fash_row = train_data[missing_positions[0]]
_, fash_hop1_top, fash_entities = find_hop1_candidates(fash_row)

print("Hop 2 후보 이름:", fash_entities)
for entity in fash_entities:
    print(entity, "→", resolve_doc_titles(entity))

좋아. 수정한 Hop 1 함수가 Al-Masjid를 후보로 찾았고, 제목 확장으로 정답 문서 Al-Masjid an-Nabawi도 찾아냈어.
동시에 mosques, Medina, Saudi Arabia 같은 후보도 생겼어. 이런 잡음이 있어도 Hop 2 문장 순위에서 정답 문장이 위로 올라오는지 확인하자. Fash 샘플 하나만 실행하면 돼.

In [ ]:
fash_hop2_ranked = find_hop2_evidence(fash_row, fash_entities)

print("Hop 2 문장 후보 상위 5개:")
for score, (title, sentence_id, sentence) in fash_hop2_ranked[:5]:
    print(f"\n점수: {score:.3f} | {title} | 문장 {sentence_id}")
    print(sentence)

Al-Masjid an-Nabawi 문서의 “Muhammad가 원래 건립했다”는 문장이

---

상위에 나오는지 보자. 이 샘플에서 확인한 뒤 후보 이름을 줄이고, 처리 속도도 개선할 거야.

여기서는 Hop 2의 2위 문장이 중요한 근거야. 그 문장이 Al-Masjid an-Nabawi가 Muhammad에 의해 세워졌다고 설명해. 반면 1위 문장은 “세 번째 모스크”라고만 하므로 claim의 해당 부분을 뒷받침하지 않아.
이유는 TF‑IDF가 단어 겹침을 점수화하기 때문이야. 점수 1위 문장이 곧 정답 근거라고 볼 수 없으니, 모델 입력에서 Hop 2 문장을 1개만 남기지 말고 상위 3개 후보를 보존하자. 아래 코드로 build_model_example을 바꾼 뒤 Fash 샘플 하나만 다시 확인해 줘.

In [ ]:
def build_model_example(example):
    _, hop1_top, entity_candidates = find_hop1_candidates(example)

    if not hop1_top:
        return None

    # Hop 2 후보 이름이 언급된 Hop 1 문장을 선택
    hop1_match = next(
        (
            item for item in hop1_top
            if any(name.lower() in item[1][2].lower()
                   for name in entity_candidates)
        ),
        hop1_top[0]
    )
    _, (hop1_title, hop1_sentence_id, hop1_sentence) = hop1_match

    hop2_ranked = find_hop2_evidence(example, entity_candidates)
    hop2_top3 = hop2_ranked[:3]

    hop2_evidence_text = "\n".join(
        f"[{title}, 문장 {sentence_id}] {sentence}"
        for _, (title, sentence_id, sentence) in hop2_top3
    )

    hop2_title = hop2_top3[0][1][0] if hop2_top3 else ""

    model_text = (
        f"Claim: {example['claim']}\n"
        f"Evidence 1: [{hop1_title}, 문장 {hop1_sentence_id}] {hop1_sentence}\n"
        f"Evidence 2 후보:\n{hop2_evidence_text}"
    )

    return {
        "text": model_text,
        "label": example["label"],
        "hop1_title": hop1_title,
        "hop2_title": hop2_title,
    }


fash_model_example = build_model_example(fash_row)
print(fash_model_example["text"])
print("\n학습 정답 라벨:", fash_model_example["label"])

출력의 Hop 2 후보에 Muhammad가 원래 건립했다는 문장이 포함되는지 보자. 이 코드는 아직 모델 학습이 아니고, 근거 후보를 성급히 하나로 줄이지 않도록 입력을 만드는 단계야.

이 결과를 보니 NOT_SUPPORTED 라벨은 모순이 아니야. 근거는 4.5km 떨어져 있다고만 하고, claim에 추가된 **동쪽(east)**이라는 방향은 말하지 않아. Muhammad가 원래 건립했다는 부분은 Hop 2 근거가 뒷받침하지만, claim의 모든 내용이 근거로 확인돼야 SUPPORTED로 볼 수 있어.
이번 샘플에서는 두 홉의 근거를 찾았지만, claim의 방향 정보가 확인되지 않아 NOT_SUPPORTED인 거야. 앞서 라벨 불일치일 수 있다고 했던 의심은 이 예시에서는 해소됐어.
다음은 전체 학습 데이터에 적용하기 전 처리 속도를 개선하는 단계야. 100개에 9분이 걸렸으니 지금 방식으로 전체를 실행하면 너무 오래 걸릴 수 있어. 문서 본문을 반복해서 읽고 문장으로 나누는 일을 캐시해 처리 시간을 줄인 뒤, 100개 시험을 다시 해보자.

좋아. 문서 제목이 여러 claim에서 반복될 때 DB를 다시 조회하고 문장을 다시 나누지 않도록 캐시를 붙이자. 캐시는 검색 결과나 라벨을 바꾸지 않고, 이미 처리한 문서의 문장 목록을 재사용해.
아래 코드를 새 Colab 셀 하나에 전부 붙여넣고 실행해 줘. 기존 함수들을 캐시를 쓰는 버전으로 바꾸고, Fash 샘플 하나로 확인하는 코드야.

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=2000)
def get_sentences(title):
    cursor.execute(
        "SELECT text FROM documents WHERE id = ?",
        (title,)
    )
    result = cursor.fetchone()

    if result is None:
        return tuple()

    return tuple(text_to_sentences(result[0]).splitlines())


@lru_cache(maxsize=5000)
def resolve_doc_titles(name, limit=5):
    cursor.execute(
        "SELECT id FROM documents WHERE id = ? COLLATE NOCASE",
        (name,)
    )
    exact = cursor.fetchone()

    if exact:
        return (exact[0],)

    cursor.execute(
        "SELECT id FROM documents WHERE id LIKE ? LIMIT ?",
        (name + "%", limit)
    )
    return tuple(row[0] for row in cursor.fetchall())


def find_hop1_candidates(example, sentence_top_k=3):
    retrieved = retrieval_by_claim.get(example["claim"])
    if retrieved is None:
        return [], [], []

    doc_titles = retrieved["doc_retrieval_results"][0][0][:3]
    sentence_candidates = []

    for title in doc_titles:
        sentences = get_sentences(title)

        for sentence_id, sentence in enumerate(sentences):
            sentence_candidates.append((title, sentence_id, sentence))

    if not sentence_candidates:
        return doc_titles, [], []

    texts = [example["claim"]] + [item[2] for item in sentence_candidates]
    matrix = TfidfVectorizer(stop_words="english").fit_transform(texts)
    scores = cosine_similarity(matrix[0:1], matrix[1:]).ravel()

    ranked_sentences = sorted(
        zip(scores, sentence_candidates),
        key=lambda item: item[0],
        reverse=True
    )

    allowed_labels = {"PERSON", "ORG", "GPE", "LOC", "FAC"}
    entity_candidates = []

    for _, (_, _, sentence) in ranked_sentences[:sentence_top_k]:
        for entity in nlp(sentence).ents:
            if (
                entity.label_ in allowed_labels
                and entity.text not in entity_candidates
            ):
                entity_candidates.append(entity.text)

    return doc_titles, ranked_sentences[:sentence_top_k], entity_candidates


def find_hop2_evidence(example, entity_candidates):
    sentence_candidates = []
    seen_titles = set()

    for entity in entity_candidates:
        for title in resolve_doc_titles(entity):
            if title in seen_titles:
                continue
            seen_titles.add(title)

            sentences = get_sentences(title)
            for sentence_id, sentence in enumerate(sentences):
                sentence_candidates.append((title, sentence_id, sentence))

    if not sentence_candidates:
        return []

    texts = [example["claim"]] + [item[2] for item in sentence_candidates]
    matrix = TfidfVectorizer(stop_words="english").fit_transform(texts)
    scores = cosine_similarity(matrix[0:1], matrix[1:]).ravel()

    return sorted(
        zip(scores, sentence_candidates),
        key=lambda item: item[0],
        reverse=True
    )


fash_row = train_data[missing_positions[0]]
fash_model_example = build_model_example(fash_row)

print("Hop 1:", fash_model_example["hop1_title"])
print("Hop 2:", fash_model_example["hop2_title"])
print("문장 캐시 상태:", get_sentences.cache_info())
print("제목 캐시 상태:", resolve_doc_titles.cache_info())

문서 캐시 상태에서 hits가 0보다 크면, 같은 제목을 재사용한 거야. 출력 결과를 확인한 다음에 100개 시험을 다시 돌릴지 정하자.


캐시가 정상 작동하고 있어. 이번 샘플에서는 문장을 한 번 재사용했고(hits=1), 문서 11개는 처음 읽었어(misses=11). 한 샘플만으로 처리 속도가 얼마나 개선됐는지는 아직 알 수 없어.

전체 100개를 다시 처리하기 전에 10개만 돌려서 새 함수의 속도와 Hop 2 문서 검색 여부를 확인하자. 아래 코드는 경과 시간도 출력해.

In [ ]:
import time
from tqdm.auto import tqdm

start_time = time.perf_counter()
train_test10 = []

for row in tqdm(train_data[:10]):
    result = build_model_example(row)
    if result is not None:
        train_test10.append(result)

elapsed = time.perf_counter() - start_time

print("처리 성공:", len(train_test10), "/ 10")
print("Hop 2 문서까지 찾음:",
      sum(bool(row["hop2_title"]) for row in train_test10), "/",
      len(train_test10))
print(f"걸린 시간: {elapsed:.1f}초")
print("문장 캐시:", get_sentences.cache_info())

이건 최종 성능 측정이 아니라 수정한 파이프라인의 속도와 동작을 점검하는 시험이야. 출력 결과를 보고 전체 데이터 처리 방법을 정하자.

이번 10개에서는 Hop 2 문서를 10개 모두 찾았어. 다만 이건 작은 동작 확인이지 최종 성능은 아니야. 처리 시간은 여전히 10개에 49초라, 그대로 전체를 돌리면 오래 걸려. 캐시도 111개 문서를 처음 읽고 21번만 재사용했으니, 캐시만으로 충분히 빨라졌다고 보긴 어려워.
우선 Hop 2에서 이름 하나가 여러 제목 후보로 확장될 때 가져오는 문서를 5개에서 2개로 줄여보자. 아래 셀을 실행한 다음, 다음 10개 샘플에서 처리 시간과 Hop 2 검색 여부를 확인해 줘.

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=5000)
def resolve_doc_titles(name, limit=2):
    cursor.execute(
        "SELECT id FROM documents WHERE id = ? COLLATE NOCASE",
        (name,)
    )
    exact = cursor.fetchone()

    if exact:
        return (exact[0],)

    cursor.execute(
        "SELECT id FROM documents WHERE id LIKE ? LIMIT ?",
        (name + "%", limit)
    )
    return tuple(row[0] for row in cursor.fetchall())


import time
from tqdm.auto import tqdm

start_time = time.perf_counter()
train_test_next10 = []

for row in tqdm(train_data[10:20]):
    result = build_model_example(row)
    if result is not None:
        train_test_next10.append(result)

elapsed = time.perf_counter() - start_time

print("처리 성공:", len(train_test_next10), "/ 10")
print("Hop 2 문서까지 찾음:",
      sum(bool(row["hop2_title"]) for row in train_test_next10), "/",
      len(train_test_next10))
print(f"걸린 시간: {elapsed:.1f}초")
print("문장 캐시:", get_sentences.cache_info())

이 시험은 속도와 검색 여부를 보는 거야. 아직 전체 데이터에 적용하거나 최종 성능으로 보고할 단계는 아니야.

이번 10개에서도 Hop 2 문서를 전부 찾았어. 다만 이 시험에서는 10개 처리에 58초가 걸렸고, 캐시도 191개 문서를 처음 읽었어. 대부분 처음 보는 문서라 반복 조회 캐시가 속도를 크게 줄이지 못한 것 같아.
현재 속도대로 전체를 처리하면 학습 데이터와 검증 데이터에 대략 14시간 이상 걸릴 수 있으니, 전체 실행은 잠시 미루자. 먼저 Wikipedia DB가 Google Drive에서 읽히는지 확인하자. Drive에 있으면 로컬 저장소로 복사하는 것만으로도 조회가 빨라질 수 있어.
아래를 실행해 줘.

In [ ]:
cursor.execute("PRAGMA database_list")

for row in cursor.fetchall():
    print("DB 이름:", row[1])
    print("DB 파일 경로:", row[2])

DB 경로가 /content/hover/wiki_wo_links.db라서 Google Drive가 아니라 Colab의 로컬 저장소에 있어. 따라서 Drive에서 읽는 속도 문제는 아니야. 전체 데이터 처리는 아직 미루고, 샘플 하나를 프로파일링해서 어느 부분에 시간이 많이 드는지 확인하자.
아래 셀은 문서 조회, 문장 분리, TF‑IDF 계산, spaCy 분석 중 시간이 많이 걸리는 부분을 보여줘. 캐시를 비우고 실행하므로 첫 조회에 가까운 시간을 측정해.

In [ ]:
import cProfile
import pstats

get_sentences.cache_clear()
resolve_doc_titles.cache_clear()

profiler = cProfile.Profile()
profiler.enable()

_ = build_model_example(train_data[20])

profiler.disable()

stats = pstats.Stats(profiler)
stats.sort_stats("cumtime").print_stats(15)

프로파일링 결과에서 병목이 보였어. TF‑IDF와 spaCy는 빠르고, 제목을 찾는 resolve_doc_titles가 약 3.65초를 썼어. 현재 함수의 대소문자 무시 검색(COLLATE NOCASE)과 LIKE가 523만 개 제목을 훑고 있을 가능성이 커. 전체 실행은 아직 하지 말자.
먼저 SQLite가 제목 검색에 인덱스를 쓰는지 확인하자. 아래 셀을 실행해 줘.

In [ ]:
queries = [
    (
        "정확히 일치, 대소문자 무시",
        "EXPLAIN QUERY PLAN "
        "SELECT id FROM documents "
        "WHERE id = 'Al-Masjid' COLLATE NOCASE LIMIT 1"
    ),
    (
        "접두어 LIKE",
        "EXPLAIN QUERY PLAN "
        "SELECT id FROM documents "
        "WHERE id LIKE 'Al-Masjid%' LIMIT 5"
    ),
    (
        "접두어 GLOB",
        "EXPLAIN QUERY PLAN "
        "SELECT id FROM documents "
        "WHERE id GLOB 'Al-Masjid*' LIMIT 5"
    ),
]

for name, query in queries:
    cursor.execute(query)
    print(f"\n[{name}]")
    for row in cursor.fetchall():
        print(row)

출력에 SCAN documents가 나오면 전체 테이블을 훑는 검색이야. SEARCH documents USING ...처럼 나오면 인덱스를 사용하는 검색이야. 결과를 보고 제목 검색 함수를 고치자.

확인됐어. 정확히 일치하는 대소문자 무시 검색과 LIKE는 인덱스를 사용하지 않고, GLOB 접두어 검색은 인덱스를 사용해. 앞서 느렸던 제목 검색이 병목이라는 게 확인된 거야.
resolve_doc_titles를 아래 코드로 교체해 줘. 정확히 같은 제목을 먼저 찾고, 못 찾으면 인덱스를 쓰는 GLOB 접두어 검색을 하도록 바꾼 버전이야.

In [ ]:
@lru_cache(maxsize=5000)
def resolve_doc_titles(name, limit=2):
    # 제목 전체가 정확히 일치하는지 인덱스로 검색
    cursor.execute(
        "SELECT id FROM documents WHERE id = ? LIMIT 1",
        (name,)
    )
    exact = cursor.fetchone()

    if exact:
        return (exact[0],)

    # 정확히 일치하는 제목이 없으면 접두어로 검색
    cursor.execute(
        "SELECT id FROM documents WHERE id GLOB ? LIMIT ?",
        (name + "*", limit)
    )
    return tuple(row[0] for row in cursor.fetchall())

실행한 뒤, 같은 샘플 하나로 시간을 다시 재보자.

In [ ]:
import time

get_sentences.cache_clear()
resolve_doc_titles.cache_clear()

start_time = time.perf_counter()
profile_test = build_model_example(train_data[20])
elapsed = time.perf_counter() - start_time

print(f"처리 시간: {elapsed:.2f}초")
print("Hop 1:", profile_test["hop1_title"])
print("Hop 2:", profile_test["hop2_title"])

처리 시간: 0.04초
Hop 1: Peter Lockyer
Hop 2: Peter Lockyer

In [ ]:
def find_hop2_evidence(example, entity_candidates, exclude_titles=None):
    exclude_titles = set(exclude_titles or [])
    sentence_candidates = []
    seen_titles = set()

    for entity in entity_candidates:
        for title in resolve_doc_titles(entity):
            if title in seen_titles or title in exclude_titles:
                continue
            seen_titles.add(title)

            sentences = get_sentences(title)
            for sentence_id, sentence in enumerate(sentences):
                sentence_candidates.append((title, sentence_id, sentence))

    if not sentence_candidates:
        return []

    texts = [example["claim"]] + [item[2] for item in sentence_candidates]
    matrix = TfidfVectorizer(stop_words="english").fit_transform(texts)
    scores = cosine_similarity(matrix[0:1], matrix[1:]).ravel()

    return sorted(
        zip(scores, sentence_candidates),
        key=lambda item: item[0],
        reverse=True
    )


def build_model_example(example):
    _, hop1_top, entity_candidates = find_hop1_candidates(example)

    if not hop1_top:
        return None

    hop1_match = next(
        (
            item for item in hop1_top
            if any(name.lower() in item[1][2].lower()
                   for name in entity_candidates)
        ),
        hop1_top[0]
    )
    _, (hop1_title, hop1_sentence_id, hop1_sentence) = hop1_match

    hop2_ranked = find_hop2_evidence(
        example,
        entity_candidates,
        exclude_titles={hop1_title}
    )
    hop2_top3 = hop2_ranked[:3]

    hop2_evidence_text = "\n".join(
        f"[{title}, 문장 {sentence_id}] {sentence}"
        for _, (title, sentence_id, sentence) in hop2_top3
    )

    hop2_title = hop2_top3[0][1][0] if hop2_top3 else ""

    model_text = (
        f"Claim: {example['claim']}\n"
        f"Evidence 1: [{hop1_title}, 문장 {hop1_sentence_id}] {hop1_sentence}\n"
        f"Evidence 2 후보:\n{hop2_evidence_text}"
    )

    return {
        "text": model_text,
        "label": example["label"],
        "hop1_title": hop1_title,
        "hop2_title": hop2_title,
    }


for name, example in [
    ("일반 샘플", train_data[20]),
    ("Fash 샘플", fash_row),
]:
    result = build_model_example(example)
    print(f"\n[{name}]")
    print("Claim:", example["claim"])
    print("Hop 1:", result["hop1_title"])
    print("Hop 2:", result["hop2_title"] or "다른 문서를 찾지 못함")
    print("라벨:", result["label"])

Hop 2가 Hop 1과 다른 문서로 나오는지 확인하자. Fash 샘플은 Hop 1에서 Fas'h mosque, Hop 2에서 Al-Masjid an-Nabawi가 나오는지 살펴보면 돼.

좋아. 이제 Hop 1과 Hop 2가 서로 다른 문서로 연결되는 걸 두 샘플에서 확인했어. Fash 샘플의 라벨도 설명돼: 두 근거는 거리와 건립자를 뒷받침하지만, 첫 근거에는 동쪽이라는 방향이 없어.
제목 검색을 빠르게 바꾼 뒤 전체 처리 속도가 얼마나 나아졌는지 아직 확인하지 않았어. 캐시를 비우고 새 함수로 학습 데이터 100개만 다시 시험하자. 이건 파이프라인 점검이지, 최종 성능 측정은 아니야.

In [ ]:
get_sentences.cache_clear()
resolve_doc_titles.cache_clear()

import time
from tqdm.auto import tqdm
from collections import Counter

start_time = time.perf_counter()
train_preview_fast = []
errors = []

for row in tqdm(train_data[:100]):
    try:
        result = build_model_example(row)
        if result is not None:
            train_preview_fast.append(result)
    except Exception as e:
        errors.append((row["uid"], str(e)))

elapsed = time.perf_counter() - start_time

print("처리 성공:", len(train_preview_fast), "/ 100")
print("Hop 2 문서까지 찾음:",
      sum(bool(row["hop2_title"]) for row in train_preview_fast), "/",
      len(train_preview_fast))
print("오류:", len(errors))
print(f"걸린 시간: {elapsed:.1f}초")
print("라벨 분포:",
      Counter(row["label"] for row in train_preview_fast))

if errors:
    print("첫 오류:", errors[0])

좋아. 새 함수는 100개를 3.8초에 처리했고, 그중 98개에서 Hop 2 문서까지 찾았어. 이건 파이프라인 점검 결과야. 98/100은 두 번째 문서를 찾았다는 뜻이지, 정답 근거를 모두 찾았거나 라벨을 맞혔다는 뜻은 아니야.
이제 학습용 7,241개와 검증용 1,811개 전체에 같은 처리를 적용하자. 앞선 속도라면 몇 분 정도 걸릴 것으로 보여. 아래 코드를 실행해 줘.

In [ ]:
from tqdm.auto import tqdm
from collections import Counter

def build_dataset(rows, split_name):
    results = []
    errors = []

    for row in tqdm(rows, desc=split_name):
        try:
            result = build_model_example(row)

            if result is None:
                errors.append((row["uid"], "모델 입력을 만들지 못함"))
            else:
                results.append(result)

        except Exception as e:
            errors.append((row["uid"], str(e)))

    return results, errors


train_model_data, train_errors = build_dataset(train_data, "학습 데이터")
val_model_data, val_errors = build_dataset(val_data, "검증 데이터")

print("\n학습 입력 생성:", len(train_model_data), "/", len(train_data))
print("학습 Hop 2 문서 발견:",
      sum(bool(row["hop2_title"]) for row in train_model_data))
print("학습 오류:", len(train_errors))
print("학습 라벨 분포:",
      Counter(row["label"] for row in train_model_data))

print("\n검증 입력 생성:", len(val_model_data), "/", len(val_data))
print("검증 Hop 2 문서 발견:",
      sum(bool(row["hop2_title"]) for row in val_model_data))
print("검증 오류:", len(val_errors))
print("검증 라벨 분포:",
      Counter(row["label"] for row in val_model_data))

if train_errors:
    print("\n학습 첫 오류:", train_errors[0])
if val_errors:
    print("검증 첫 오류:", val_errors[0])

좋아. 전체 입력 생성이 끝났고 오류도 없어.
- 학습 데이터에서 Hop 2 문서 발견: 7,004 / 7,241 (96.7%)
- 검증 데이터에서 Hop 2 문서 발견: 1,731 / 1,811 (95.6%)
이 비율은 Hop 2 문서를 후보로 찾았다는 뜻이야. 정답 근거 문장을 찾았거나 최종 라벨을 맞혔다는 성능은 아직 아니야.
이제 앞서 정한 첫 판정 모델인 TF‑IDF + Logistic Regression을 학습하고 검증하자. TF‑IDF가 claim과 근거 문장을 숫자 벡터로 바꾸고, Logistic Regression이 SUPPORTED와 NOT_SUPPORTED를 분류해. 라벨 비율이 약 72:28로 차이가 있으니, 두 라벨을 모두 고려하도록 class_weight="balanced"를 사용하자.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from collections import Counter

X_train = [row["text"] for row in train_model_data]
y_train = [row["label"] for row in train_model_data]

X_val = [row["text"] for row in val_model_data]
y_val = [row["label"] for row in val_model_data]

model = make_pipeline(
    TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        min_df=2,
        max_features=100_000,
        sublinear_tf=True
    ),
    LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    )
)

model.fit(X_train, y_train)
predictions = model.predict(X_val)

labels = ["SUPPORTED", "NOT_SUPPORTED"]

print("Accuracy:", round(accuracy_score(y_val, predictions), 4))
print("Macro-F1:", round(f1_score(y_val, predictions, average="macro"), 4))

print("\n라벨별 결과:")
print(classification_report(
    y_val,
    predictions,
    labels=labels,
    zero_division=0
))

print("혼동 행렬 (실제 행 × 예측 열, 순서: SUPPORTED, NOT_SUPPORTED):")
print(confusion_matrix(y_val, predictions, labels=labels))

majority_label = Counter(y_train).most_common(1)[0][0]
majority_predictions = [majority_label] * len(y_val)
print(
    "\n다수 라벨만 예측하는 기준 Accuracy:",
    round(accuracy_score(y_val, majority_predictions), 4)
)

Accuracy는 전체 정답률이고, Macro-F1은 두 라벨의 F1을 같은 비중으로 평균해. 라벨 수가 다른 데이터에서는 둘 다 보고 비교하는 게 좋아. 이 검증 결과는 현재 파이프라인과 80:20 분할 기준의 baseline 수치가 될 거야.

이제 baseline v1의 분류 성능을 얻었어.
- Accuracy 0.5903
- Macro-F1 0.4931
다수 라벨인 SUPPORTED만 예측해도 Accuracy는 0.7178이야. 다만 그 방식의 Macro-F1은 약 0.418이라, 현재 모델은 Accuracy는 낮지만 소수 라벨인 NOT_SUPPORTED도 일부 찾아내고 있어. 실제로 NOT_SUPPORTED 511개 중 138개를 맞혔어.
혼동 행렬은 이렇게 읽으면 돼.
실제 라벨	SUPPORTED 예측	NOT_SUPPORTED 예측
SUPPORTED	931	369
NOT_SUPPORTED	373	138


아직 모델을 조정하기 전에, 어떤 사례에서 틀리는지 먼저 살펴보자. 다음 코드는 NOT_SUPPORTED를 SUPPORTED로 잘못 판정한 사례와 반대 사례를 각각 3개씩 보여줘.

In [ ]:
def show_errors(gold_label, predicted_label, limit=3):
    shown = 0

    for i, (gold, pred) in enumerate(zip(y_val, predictions)):
        if gold == gold_label and pred == predicted_label:
            print("\nClaim:", val_data[i]["claim"])
            print("실제:", gold, "| 예측:", pred)
            print("Hop 1:", val_model_data[i]["hop1_title"])
            print("Hop 2:", val_model_data[i]["hop2_title"] or "문서 없음")
            print("모델 입력과 근거 후보:")
            print(val_model_data[i]["text"][:1200])

            shown += 1
            if shown >= limit:
                break


print("실제 NOT_SUPPORTED인데 SUPPORTED로 예측한 사례")
show_errors("NOT_SUPPORTED", "SUPPORTED")

print("\n실제 SUPPORTED인데 NOT_SUPPORTED로 예측한 사례")
show_errors("SUPPORTED", "NOT_SUPPORTED")


오류 사례를 보면 baseline의 약점이 분명해졌어요. 분류기가 틀린 경우도 있지만, 그 전에 검색된 근거가 claim과 맞지 않는 경우도 있습니다. 예를 들어 Fanny Talbot 사례는 Hop 1이 Fanny Talbot이 아니라 Brian Sewell로 잡혔고, Hop 2 문장 후보에는 John Ruskin 근거가 들어갔지만 claim의 “그 남자”가 누구인지 연결을 잘못했습니다. Liverpool 사례는 Hop 2 검색 결과가 주장 속 Hugh Clifford와 무관한 문장들이었습니다.

반면 Pietro Ferrero 사례에서는 Hop 1 문장만으로도 Nutella가 1965년에 처음 소개됐다는 정보가 확인됩니다. 이 사례는 근거가 입력에 있는데도 분류기가 NOT_SUPPORTED를 예측했으므로 분류기 자체의 한계를 보여줍니다.

따라서 지금 결과는 “모델이 HoVer의 2-hop 추론을 잘한다”기보다, 현재 검색 절차와 TF-IDF + Logistic Regression 조합의 첫 기준점으로 해석해야 합니다. 이 기준점은 Accuracy 0.5903, Macro-F1 0.4931입니다. 다수 라벨만 예측하는 기준보다 Accuracy는 낮고 Macro-F1은 높으므로, 두 지표를 함께 보고해야 합니다.

다음으로는 모델을 바로 바꾸기보다 아래 두 가지를 확인하면 됩니다.

검색 단계 평가: 검증 샘플의 Hop 1·Hop 2 문서와 문장 후보에 정답 supporting fact가 얼마나 포함되는지 측정합니다. 정답 근거가 후보에 없으면 분류기가 맞히기 어렵습니다.
분류 단계 기준선 비교: 같은 검증 데이터에서 SUPPORTED만 예측하는 단순 기준선도 Macro-F1을 계산해 함께 비교합니다. 현재 혼동행렬로 계산하면 약 0.417이며, Logistic Regression의 0.493보다 낮습니다.
이 두 결과를 정리하면 발표에서 “검색이 어디서 실패하는지”와 “분류기가 어느 정도 개선하는지”를 구분해 설명할 수 있어요.


지금은 검색 결과 분석을 먼저 하면 돼요. 방금 보여준 오답들을 보면, 분류기가 판단하기 전에 정답 근거가 후보에 들어왔는지부터 확인해야 해요.

Colab에서 검증 데이터 전체를 훑어 각 샘플의 HoVer 정답 supporting_facts가 우리가 만든 모델 입력에 포함됐는지 세어봅시다. 이 단계에서는 정답 라벨을 모델 입력에 넣지 않고, 성능 평가용으로만 정답 근거와 검색 결과를 비교합니다.

먼저 아래 셀을 실행해 주세요. train_data 같은 앞서 쓰던 변수는 사용하지 않고, 가지고 있는 변수 이름을 확인하는 코드예요.



In [ ]:
print("검증 데이터 변수:", "val_data" in globals())
print("모델 입력 변수:", "val_model_data" in globals())

print("검증 샘플 예시 필드:", list(val_data[0].keys()))
print("모델 입력 예시 필드:", list(val_model_data[0].keys()))

좋아요. 두 데이터 모두 있고, 샘플 순서도 맞춰 비교할 수 있어요. 먼저 정답 근거 문서 제목이 모델 입력에 포함됐는지 확인해 봅시다. 아직 문장까지 맞았는지는 세지 않고, 검색이 올바른 문서를 가져왔는지만 보는 단계예요.
아래 셀을 실행해 주세요.

In [ ]:
# 첫 샘플의 정답 문서와 모델 입력에 들어간 문서를 나란히 확인
i = 0

print("Claim:", val_data[i]["claim"])
print("정답 근거:", val_data[i]["supporting_facts"])
print("Hop 1 문서:", val_model_data[i]["hop1_title"])
print("Hop 2 문서:", val_model_data[i]["hop2_title"])
print("\n모델 입력:")
print(val_model_data[i]["text"])

예시에서는 정답 문서 두 개가 모두 모델 입력에 있네요.
- Fanny Talbot은 Hop 1 문서로는 선택되지 않았지만, Hop 2 근거 후보에는 포함됐어요.
- John Ruskin은 Hop 2 문서와 근거 후보에 포함됐어요.
즉, 정답 문서가 후보에 있었는데도 Hop 1에서 잘못된 문장을 고른 경우예요. 이걸 전체 검증 데이터에서도 세어봅시다. 아래 셀은 각 정답 근거 문서 제목이 해당 샘플의 모델 입력에 들어 있는지 확인합니다.

In [ ]:
found_facts = 0
total_facts = 0
samples_with_any = 0
samples_with_all = 0

for row, model_row in zip(val_data, val_model_data):
    model_text = model_row["text"]

    # 중복된 정답 문서는 한 번만 셈
    gold_titles = set(title for title, sentence_id in row["supporting_facts"])
    total_facts += len(gold_titles)

    found_titles = {
        title for title in gold_titles
        if f"[{title}," in model_text
    }

    found_facts += len(found_titles)

    if found_titles:
        samples_with_any += 1
    if found_titles == gold_titles:
        samples_with_all += 1

print("검증 샘플 수:", len(val_data))
print(f"정답 문서 전체 중 모델 입력에 포함: {found_facts}/{total_facts} ({found_facts / total_facts:.1%})")
print(f"정답 문서를 하나 이상 포함한 샘플: {samples_with_any}/{len(val_data)} ({samples_with_any / len(val_data):.1%})")
print(f"정답 문서를 모두 포함한 샘플: {samples_with_all}/{len(val_data)} ({samples_with_all / len(val_data):.1%})")

결과가 나왔어요. 검증 샘플 1,811개 중 정답 문서를 하나 이상 찾은 경우는 80.0%, 정답 문서를 **모두 찾은 경우는 24.8%**예요. 두-hop 모두 맞아야 하는 과제라, 두 정답 문서를 모두 가져온 비율이 특히 중요합니다.
다만 이 수치는 문서 제목이 모델 입력에 들어갔는지만 확인한 결과예요. 정답 문장까지 후보로 들어갔는지는 아직 모릅니다. 다음에는 전체 샘플에서 정답 근거 문장이 입력에 있는지 확인할게요. 현재 입력은 문서명과 문장 번호를 함께 표시하므로, 아래 코드로 문서·문장 쌍을 비교할 수 있습니다.

In [ ]:
found_facts = 0
total_facts = 0
samples_with_any = 0
samples_with_all = 0

for row, model_row in zip(val_data, val_model_data):
    model_text = model_row["text"]
    gold_facts = set(
        (title, sentence_id)
        for title, sentence_id in row["supporting_facts"]
    )

    total_facts += len(gold_facts)

    found = {
        (title, sentence_id)
        for title, sentence_id in gold_facts
        if f"[{title}, 문장 {sentence_id}]" in model_text
    }

    found_facts += len(found)

    if found:
        samples_with_any += 1
    if found == gold_facts:
        samples_with_all += 1

print("검증 샘플 수:", len(val_data))
print(f"정답 문장 전체 중 모델 입력에 포함: {found_facts}/{total_facts} ({found_facts / total_facts:.1%})")
print(f"정답 문장을 하나 이상 포함한 샘플: {samples_with_any}/{len(val_data)} ({samples_with_any / len(val_data):.1%})")
print(f"정답 문장을 모두 포함한 샘플: {samples_with_all}/{len(val_data)} ({samples_with_all / len(val_data):.1%})")

수치가 나왔어요. 다만 정답 문장 수의 분모가 앞 결과와 달라서 먼저 확인이 필요해요. 문서 평가는 정답 문서 중복을 제거해 세어서 3,622였는데, 이번 문장 평가는 4,414개예요. supporting_facts에 같은 문서·문장 쌍이 반복된 샘플이 있을 수 있습니다.
샘플 하나만 확인해 주세요.

In [ ]:
i = 0
print("원본 supporting_facts:", val_data[i]["supporting_facts"])
print("중복 제거 후:", set(
    (title, sentence_id)
    for title, sentence_id in val_data[i]["supporting_facts"]
))

결과를 보면 분모 차이가 중복 때문인지 확인할 수 있어요. 어느 쪽이든 현재 계산에서 중요한 결과는 **정답 문장을 하나라도 입력에 포함한 샘플 72.4%, 모든 정답 문장을 포함한 샘플 16.0%**라는 점이에요.

이 샘플에는 중복 정답 문장이 없네요. 분모가 다른 건 세는 대상이 달라서예요.
- 문서 평가는 같은 문서에서 정답 문장이 여러 개여도 문서 제목은 한 번만 셉니다.
- 문장 평가는 정답 문장을 각각 셉니다. 그래서 정답 문장 수가 정답 문서 수보다 많을 수 있어요.
따라서 4,414는 중복을 제거한 정답 문장 수이고, 3,622는 문서 제목 수라 서로 달라도 괜찮습니다. 지금 결과는 **정답 문장이 하나 이상 입력된 샘플 72.4%, 모든 정답 문장이 입력된 샘플 16.0%**로 읽으면 됩니다.
이제 검색과 분류의 영향을 나눠 보겠습니다. 정답 문장이 모두 입력된 샘플과 하나라도 빠진 샘플에서 분류 성능을 비교하면, 근거 검색이 분류 결과에 어떤 영향을 주는지 볼 수 있어요. 아래 셀을 실행해 주세요.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

complete_indices = []
incomplete_indices = []

for i, (row, model_row) in enumerate(zip(val_data, val_model_data)):
    model_text = model_row["text"]
    gold_facts = set(
        (title, sentence_id)
        for title, sentence_id in row["supporting_facts"]
    )
    found_facts = {
        (title, sentence_id)
        for title, sentence_id in gold_facts
        if f"[{title}, 문장 {sentence_id}]" in model_text
    }

    if found_facts == gold_facts:
        complete_indices.append(i)
    else:
        incomplete_indices.append(i)

for name, indices in [
    ("정답 문장을 모두 찾은 샘플", complete_indices),
    ("정답 문장이 빠진 샘플", incomplete_indices),
]:
    gold = [y_val[i] for i in indices]
    pred = [predictions[i] for i in indices]

    print(f"\n{name}: {len(indices)}개")
    print("Accuracy:", round(accuracy_score(gold, pred), 4))
    print("Macro-F1:", round(f1_score(
        gold, pred,
        labels=["SUPPORTED", "NOT_SUPPORTED"],
        average="macro",
        zero_division=0
    ), 4))

이 결과는 정답 문장을 모두 입력에 넣은 샘플에서 성능이 조금 더 높았다는 뜻이에요.
샘플 구분	개수	Accuracy	Macro-F1
정답 문장을 모두 찾음	289	0.6194	0.5283
정답 문장이 일부 빠짐	1,522	0.5848	0.4864


차이는 Accuracy 3.46%p, Macro-F1 4.19%p예요. 검색이 잘되면 분류에 도움이 되지만, 정답 문장을 모두 찾은 경우에도 Macro-F1이 0.5283이므로 검색뿐 아니라 분류기도 근거의 의미와 claim의 관계를 잘 판단하지 못하는 사례가 남아 있다고 볼 수 있습니다.
이제 첫 baseline의 평가와 오류 분석은 갖춰졌어요. 발표에는 전체 성능(Accuracy 0.5903, Macro-F1 0.4931), 다수 라벨 기준, 정답 문장 검색률(16.0%), 그리고 방금 비교 결과를 함께 제시하면 됩니다. 이 baseline을 팀원들의 방식과 비교할 공통 기준으로 고정하고, 다음 모델을 시험할 때는 데이터 분할과 검증 샘플을 그대로 유지하세요.

In [ ]:
sample = next(
    (
        row for row in val_data
        if "Fergie" in row["claim"] and "Life Goes On" in row["claim"]
    ),
    None
)
a
if sample is None:
    print("검증 데이터에서 해당 Fergie claim을 찾지 못했어요.")
else:
    result = build_model_example(sample)

    if result is None:
        print("이 claim의 사전 검색 결과를 찾지 못했어요.")
    else:
        print("Claim:", sample["claim"])
        print("Hop 1:", result["hop1_title"])
        print("Hop 2:", result["hop2_title"] or "찾지 못함")
        print("\n근거 후보:")
        print(result["text"])

        prediction = clf.predict(
            vectorizer.transform([result["text"]])
        )[0]
        print("\n예측 결과:", prediction)
        print("HoVer 정답 라벨:", sample["label"])

학습 데이터의 supporting_fact가 DB에서 실제 문장으로 잘 연결되는지 확인하는 것이이에요. 이 문장들 이 reranker에게 이 claim의 정단 근거라고 알려줄 학습 예시가 됩니다.

In [ ]:
for name, value in list(globals().items()):
    if isinstance(value, list) and value and isinstance(value[0], dict):
        keys = value[0].keys()
        if "claim" in keys and "supporting_facts" in keys:
            print(f"변수명: {name} | 샘플 수: {len(value)}")